# Ducati: Guida a Datapizza AI

Benvenuti in questo tutorial interattivo. Esploreremo le funzionalità principali per costruire sistemi AI sfruttando la semplicità e la personalizzabilità del framework `datapizza-ai`:
1.  **Chiamate API**: Come interrogare un modello.
2.  **Memoria**: Come costruire un chatbot che ricorda il contesto.
3.  **Tools**: Come dare superpoteri al modello (meteo, calcoli, ecc.).

## 1. Setup Iniziale

Importiamo le librerie necessarie e carichiamo le variabili d'ambiente (come la API Key). Assicurati di avere un file `.env` nella cartella principale con `OPENAI_API_KEY=...`.

In [3]:
import os
from dotenv import load_dotenv

# Carica le variabili d'ambiente (API KEY)
load_dotenv()

True

## 2. Il Client e la prima chiamata API

Il cuore di un sistema di AI sono i **Client** e le chiamate API. In questo esempio usiamo l'`OpenAIClient`, ma la libreria supporta anche Anthropic, Google, Mistral, ecc., mantenendo la stessa interfaccia.

Una chiamata API significa mandare la richiesta dell'utente ad un provider esterno (in questo caso OpenAI). Questo è reso possibile attraverso il client.

### Parametri chiave:
*   **`model`**: Il modello specifico da usare (es. `gpt-4o`, `gpt-5.1`).
*   **`temperature`**: Controlla la "creatività". Valori bassi (es. 0.2) rendono il modello più deterministico e preciso; valori alti (es. 1.5) lo rendono più vario e creativo.

Proviamo a inizializzare il client e fare una domanda diretta.

In [4]:
from datapizza.clients.openai import OpenAIClient

# Inizializzazione del Client
client = OpenAIClient(
    api_key=os.getenv("OPENAI_API_KEY"),
    model="gpt-5.1", 
    temperature=1.7 # Bilanciato tra creatività e coerenza
)

# Chiamata diretta (stateless)
print("--- Invio richiesta ---")
response = client.invoke("Spiega in una barzelletta sulla MotoGP.")

print(f"Risposta: {response.text}")

--- Invio richiesta ---
Risposta: Al Mugello, due moto si fermano ai box.

La prima sospira:  
«Sono distrutta, il pilota continua a piegare come un pazzo… durerò ancora un paio di giri e poi sono da buttare!»

E l’altra:  
«Beata te… il mio frena venti metri *dopo* il cartello dei 100… io non so che gomme ho, ma di sicuro lui non ha il cervello.»


## 3. Gestione della Memoria (Chatbot)

Le API degli LLM sono di base *stateless*: non ricordano cosa è stato detto nella richiesta precedente. Per creare una chat, dobbiamo gestire manualmente la cronologia.

### Oggetti Fondamentali:
*   **`Memory`**: È il contenitore che memorizza la lista dei messaggi scambiati.
*   **`ROLE`**: Un'enumerazione che definisce CHI sta parlando:
    *   `ROLE.USER`: L'utente umano.
    *   `ROLE.ASSISTANT`: La risposta dell'AI.
    *   `ROLE.SYSTEM`: Istruzioni nascoste per definire il comportamento del bot.
*   **`TextBlock`**: Rappresenta il contenuto testuale di un messaggio. (Nota: `datapizza-ai` supporta anche `ImageBlock` per input multimodali!).

Costruiamo una classe `Chatbot` che usa questi oggetti per mantenere il filo del discorso.

In [5]:
from datapizza.memory import Memory
from datapizza.type import ROLE, TextBlock

class Chatbot:
    def __init__(self, client):
        self.client = client
        # Partiamo da 0, inizializziamo una memoria vuota per questa sessione
        self.memory = Memory()

    def send(self, user_input: str) -> str:
        # 1. Aggiungiamo il messaggio dell'utente alla memoria
        self.memory.add_turn(
            [TextBlock(content=user_input)], 
            ROLE.USER
        )
        
        # 2. Invochiamo il client passando l'intera memoria accumulata
        response = self.client.invoke(user_input, memory=self.memory)
        
        # 3. Salviamo la risposta nella memoria per il futuro
        self.memory.add_turn(
            [TextBlock(content=response.text)], 
            ROLE.ASSISTANT
        )
        
        # Metriche opzionali
        total_tokens = (response.prompt_tokens_used or 0) + (response.completion_tokens_used or 0)
        print(f"[Debug] Token usati in questo turno: {total_tokens}")
        
        return response.text

### Test Interattivo
Esegui la cella qui sotto per chattare. Nota come il bot ricorderà il tuo nome o le informazioni che gli dai man mano che la conversazione prosegue.

In [6]:
# Usiamo lo stesso client creato sopra
bot = Chatbot(client)

print("Buongiorno! Digita 'esci' per terminare.")

while True:
    try:
        user = input("Tu > ").strip()
        if user.lower() in {"esci", "exit", "quit"}:
            print("Arrivederci!")
            break
        if not user:
            continue
            
        # Il metodo send gestisce tutto il flusso
        reply = bot.send(user)
        print(f"Bot > {reply}")
        
    except KeyboardInterrupt:
        print("\nChat interrotta.")
        break
    except Exception as e:
        print(f"Errore: {e}")
        break


Buongiorno! Digita 'esci' per terminare.


Tu >  Cia, mi chiam Mirk, cm ti chiami?


[Debug] Token usati in questo turno: 80
Bot > Ciao Mirko, piacere!  
Io sono un’intelligenza artificiale, puoi chiamarmi semplicemente **Assistente**.  

Come posso aiutarti?


Tu >  Cia assistente, cm vedi nn riesc a scrivere una vcale, qual è?


[Debug] Token usati in questo turno: 209
Bot > La vocale che manca nei tuoi messaggi è la **“o”**.

Esempi con la “o” inserita:
- *Cia* → **Ciao**  
- *Mirk* → **Mirko**  
- *cm* → **come**  
- *nn riesc* → **non riesco**  

Se vuoi, posso aiutarti a riscrivere le frasi in modo corretto.


Tu >  na nn serve, ti ricrdi cm mi chiam?


[Debug] Token usati in questo turno: 235
Bot > Sì, certo: ti chiami **Mirko**.


Tu >  exit


Arrivederci!


## 4. Tools (Function Calling)

I modelli linguistici sono potenti ma "chiusi" nel loro training set. Non sanno che ore sono adesso, non possono calcolare radici quadrate complesse e non sanno che tempo fa a Bologna.

I **Tools** risolvono questo problema. Definiamo funzioni Python che il modello può decidere di "chiamare".

### Il Decoratore `@tool`
Usa `@tool` per trasformare una funzione normale in uno strumento che il modello può capire. È importante scrivere una **docstring** chiara, perché il modello la legge per capire *quando* e *come* usare il tool.

### `tool_choice`
Quando invochiamo il client, possiamo guidare il comportamento:
*   `"auto"` (default): Il modello decide se usare un tool o rispondere normalmente.
*   `"required"`: Forza il modello a usare un tool.
*   `"none"`: Impedisce l'uso dei tool.

In [7]:
from datapizza.tools import tool
import datetime

# 1. Definizione dei Tool

@tool
def get_current_time() -> str:
    """Restituisce l'ora e la data correnti."""
    now = datetime.datetime.now()
    return now.strftime("%Y-%m-%d %H:%M:%S")

@tool
def get_weather(city: str) -> str:
    """Restituisce il meteo corrente per una data città."""
    # Simulazione: qui potresti chiamare un'API reale (OpenWeatherMap, ecc.)
    return f"A {city} c'è il sole e ci sono 25°C (Simulato)"

@tool
def calculate(expression: str) -> str:
    """Esegue un calcolo matematico. Input: una stringa matematica (es. '12 * 5')."""
    try:
        # Nota: eval è rischioso in produzione, usare con cautela
        return str(eval(expression))
    except Exception as e:
        return f"Errore nel calcolo: {e}"

print("Tools definiti: Meteo, Orario, Calcolatrice.")

Tools definiti: Meteo, Orario, Calcolatrice.


### Esecuzione con Tools
Ora facciamo una domanda che richiede l'uso di questi strumenti. Notate come passiamo la lista `tools=[...]` al metodo `invoke`.

Se il modello usa un tool, `datapizza-ai` intercetta la richiesta, esegue la funzione Python corrispondente e restituisce il risultato.

In [8]:
# Chiediamo qualcosa che richiede i tool
prompt = "Che ore sono adesso e quanto fa 150 diviso 3?"
print(f"Prompt utente: '{prompt}'\n")

response = client.invoke(
    prompt,
    tools=[get_current_time, calculate, get_weather],
    tool_choice="auto"
)

print(f"Risposta finale del modello:\n{response.text}")

# Vediamo infine cosa è successo "dietro le quinte"
if hasattr(response, 'function_calls') and response.function_calls:
    print("\n--- Debug: Chiamate ai Tool eseguite ---")
    for call in response.function_calls:
        # Eseguiamo effettivamente il tool per mostrare il risultato raw
        # Nota: client.invoke lo fa già automaticamente per generare response.text,
        # qui lo rifacciamo solo per didattica o per mostrare i parametri scelti dal modello.
        print(f"Tool chiamato: {call.name}")
        print(f"Argomenti: {call.arguments}")

Prompt utente: 'Che ore sono adesso e quanto fa 150 diviso 3?'

Risposta finale del modello:


--- Debug: Chiamate ai Tool eseguite ---
Tool chiamato: get_current_time
Argomenti: {}
Tool chiamato: calculate
Argomenti: {'expression': '150 / 3'}
